# Lift, Splat, Shoot: a from-scratch mini demo

Six cameras on a car, one top-down map out, and a planner on top.
Everything here (the world, the cameras, the model) is written in this repo, so every step can be inspected.

Run `python train.py` first (about 25 min on CPU), or use the checkpoint in `checkpoints/`.

In [ ]:
%matplotlib inline
import numpy as np, torch, matplotlib.pyplot as plt
from lss import world, model as M, plan as P, viz as V
from demo import load_model, predict, ious, rotate_yaw, TEST_SEED

net = load_model("checkpoints/lss.pt")
rig = world.DEFAULT_RIG
test = world.make_batch(64, np.random.default_rng(TEST_SEED))
s = 0   # which scene to show; change it and re-run the cells

## 0. The input: six sideways views
A pixel tells you *direction*, not *distance*. The camera model is `x = R (d K^-1 [u, v, 1]) + t`, and `d` is unknown.

In [ ]:
V.plot_cameras(test["imgs"][s], rig.names); plt.show()

## 1. Lift: predict a distribution over depth for every pixel
`c(d) = alpha(d) * c`. Each feature pixel becomes a D x C block (outer product).
The dashed line is the true depth, which the model **never saw during training**.

In [ ]:
logits, alpha, geom = predict(net, test, aux=True)
probs = logits.sigmoid().numpy()
cam, cell = 0, (8, 12)       # try other cameras / cells: (row, col) on the 12 x 24 feature grid
V.plot_depth_ray(test["imgs"][s, cam], alpha[s, cam], test["depth"][s, cam], cell); plt.show()

In [ ]:
V.plot_depth_scatter(alpha, test["depth"]); plt.show()

## 2. Splat: drop every lifted point onto one top-down grid and sum
Each dot is one (pixel, depth bin) from one camera, sized by alpha(d). Rays from all six cameras land in the same grid.

In [ ]:
V.plot_lifted_points(geom[s], alpha[s], rig.names); plt.show()

Same points in 3D (drag to rotate). Black boxes are the true cars. Notice the rays continue below the ground: the model spreads its guesses, and splatting ignores height anyway.

In [ ]:
V.lifted_points_3d(geom[s], alpha[s], rig.names, scene=test["scenes"][s])

In [ ]:
V.plot_bev_pair(probs[s], test["target"][s]); plt.show()
print("IoU on", len(test["imgs"]), "test scenes:", ious(logits, test))

The paper's speed trick: sort points by cell, take one running sum (cumsum), and read each cell's total off its last point. Both versions give the same answer:

In [ ]:
with torch.no_grad():
    T = {k: torch.from_numpy(test[k][:8]) for k in ("imgs","K","R","t")}
    a, lifted = net.lift(T["imgs"], T["K"])
    g = M.frustum_points(T["K"], T["R"], T["t"], *a.shape[-2:])
    print("max |index_add - cumsum| =", (M.splat(g, lifted) - M.splat_cumsum(g, lifted)).abs().max().item())
%timeit -n 5 M.splat(g, lifted)
%timeit -n 5 M.splat_cumsum(g, lifted)

## 3. Shoot: score candidate paths on the cost map, keep the cheapest
Cost = vehicles (with a safety margin) + off-road. Red outlines are the true cars.

In [ ]:
paths, scores, best, cost = P.plan(probs[s])
V.plot_plan(probs[s], paths, scores, best, cost, target=test["target"][s]); plt.show()

## 4. It doesn't fall apart
### A camera goes down

In [ ]:
mask = torch.ones(len(test["imgs"]), rig.n); mask[:, 0] = 0      # kill the front camera
p_drop = predict(net, test, cam_mask=mask)
print("all cameras:", ious(logits, test), "  front dropped:", ious(p_drop, test))
V.plot_bev_pair(p_drop.sigmoid().numpy()[s], test["target"][s], "front camera dropped"); plt.show()

### Calibration off

In [ ]:
for sigma in [0, 1, 2, 4, 8]:
    Rn = rotate_yaw(test["R"], sigma, np.random.default_rng(sigma))
    print(f"yaw error ~{sigma} deg:", ious(predict(net, test, R=Rn), test))

### A different car: 5 cameras, wider lenses, different angles, no retraining

In [ ]:
other = world.make_batch(64, np.random.default_rng(TEST_SEED), rig=world.OTHER_RIG)
o = predict(net, other)
print("different car:", ious(o, other))
V.plot_cameras(other["imgs"][s], world.OTHER_RIG.names); plt.show()
V.plot_bev_pair(o.sigmoid().numpy()[s], other["target"][s], "never trained on this rig"); plt.show()

## 5. Does learning depth matter?
The baseline spreads every pixel evenly along its ray (alpha = 1/D) instead of predicting depth.

In [ ]:
import os
if os.path.exists("checkpoints/lss_uniform.pt"):
    base = load_model("checkpoints/lss_uniform.pt")
    pb = predict(base, test)
    print("LSS:", ious(logits, test), "  no-depth baseline:", ious(pb, test))
    V.plot_bev_pair(pb.sigmoid().numpy()[s], test["target"][s], "no-depth baseline"); plt.show()
else:
    print("train it with: python train.py --uniform-depth --out checkpoints/lss_uniform.pt")